**Where the data comes from.** `results/GA/run_N` is written by `python AI/GA/optimize.py` (headless simulator; `Time` = virtual game seconds = pieces x ms-per-piece; fitness = mean game score + 1 per piece; each run also logs `Deaths`, `AvgPieces`, `AvgLinesPerSec` and a baseline agent (`BaselineFitness`, `RelBest`, `RelAvg`)). The cells below show the newest run; to look at another one, pass `GA_DIR / "run_3" / "ga_log.csv"` (or `agent_log.csv`) instead of `latest_run(GA_DIR)`. The functions also still read the old real-time format.

In [26]:
import pandas as pd
import numpy as np

# --- project paths (work from any location) ---
from pathlib import Path


def find_project_root(start=None):
    """Walk up from the notebook's folder until the folder containing AI/ and Tetris/ is found."""
    here = Path(start or Path.cwd()).resolve()
    for d in [here, *here.parents]:
        if (d / "AI").is_dir() and (d / "Tetris").is_dir():
            return d
    raise FileNotFoundError("Open this notebook from inside the Tetris-Project folder.")


ROOT = find_project_root()
GA_DIR = ROOT / "results" / "GA"
GAUNTLET_DIR = ROOT / "results" / "gauntlet"


def latest_run(base_dir):
    """Highest-numbered run_N folder inside base_dir."""
    runs = [d for d in Path(base_dir).glob("run_*") if d.name[4:].isdigit()]
    if not runs:
        raise FileNotFoundError(f"No run_N folders in {base_dir}")
    return max(runs, key=lambda d: int(d.name[4:]))


def extract_and_show_top_agents(
    csv_path,
    top_n=3,
    survival_time=None,
    min_tetrises=2,
    fallback_by_fitness=False,
    label=None,
    survivors_only=False,
):
    # Ensure all columns are displayed in notebook output
    pd.set_option('display.max_columns', None)
    pd.set_option('display.expand_frame_repr', False)  # Prevents dataframe wrapping with many columns

    df = pd.read_csv(csv_path)
    df.dropna(how="all", inplace=True)
    print(f"\n--- {label or csv_path} ---")
    if survival_time is not None or survivors_only:
        if "BestDeaths" in df.columns:
            # simulator logs: the best agent survived the whole budget if it never died
            survived = df["BestDeaths"] == 0
        else:
            # old real-time logs: survived = reached the time limit
            survived = df["BestTime"] == survival_time
        filtered = df[survived & (df["NumTetris"] > min_tetrises)]
        if not filtered.empty:
            top_agents = filtered.sort_values(by="BestFitness", ascending=False).head(top_n)
        elif fallback_by_fitness:
            top_agents = df.sort_values(by="BestFitness", ascending=False).head(top_n)
        else:
            top_agents = pd.DataFrame()
    else:
        top_agents = df.sort_values(by="BestFitness", ascending=False).head(top_n)
    display(top_agents)  # Will now show all columns, never "..."

    weights = {
        f"a{i+1}": np.array([
            row["W1:AggHeight"], row["W2:Holes"], row["W3:Blockades"], row["W4:Bumpiness"],
            row["W5:AlmostFull"], row["W6:FillsWell"], row["W7:ClearBonus4"], row["W8:ClearBonus3"],
            row["W9:ClearBonus2"], row["W10:ClearBonus1"]
        ])
        for i, (_, row) in enumerate(top_agents.iterrows())
    }
    # Print in dictionary format for code copy-paste (no rounding/truncation):
    print(f"{label or 'agent_weights'} = {{")
    for k, v in weights.items():
        # Show full float precision, no scientific notation, no suppressing small floats
        print(
            f"    '{k}': np.array({np.array2string(v, separator=', ', precision=16, suppress_small=False, floatmode='maxprec_equal')}),"
        )
    print("}")
    return weights


In [ ]:
# Newest training run. Note: the 'best agent per generation' scores are noisy (each generation uses
# different piece sequences) - re-test the candidates below in the gauntlet before trusting them.
latest_weights = extract_and_show_top_agents(
    latest_run(GA_DIR) / "ga_log.csv",
    top_n=3,
    survivors_only=True,
    min_tetrises=0,
    fallback_by_fitness=True,
    label="latest_run_weights",
)
